# RetinaSafe — BRSET multilabel baseline + fairness audit

**Goal:** train a ResNet-50 multilabel classifier on BRSET (Brazilian Multilabel Ophthalmological Dataset) for 12 disease labels, then audit per-disease AUROC across demographic subgroups (sex, camera, age band, quality).

**Why this matters:** this is the first experiment in the project that has the structure of the actual paper contribution. Multilabel + demographics + cross-camera analysis are what BRSET unlocks; OCT and APTOS were warm-ups.

**Verified before writing this notebook (in the data-prep cells you already ran):**
- 16,266 images, 8,524 unique patients, patient-disjoint splits (5,967 / 1,278 / 1,279 patients)
- 12 active disease labels (retinal_detachment dropped — only 7 positives total)
- Every label has ≥16 positives in test, so per-label AUROC is computable
- Two cameras (Canon CR 65%, Nikon NF5050 35%) — built-in cross-camera audit axis
- Sex (60/40 F/M) and camera are the cleanest fairness axes; age has 33% NaN so secondary

**Training data plan:**
- Multilabel BCE-with-logits loss, **per-label `pos_weight`** to handle severe imbalance
- ResNet-50 ImageNet-pretrained, 12-class sigmoid head (no softmax — multilabel)
- AdamW 1e-4, cosine LR + 2-epoch warmup, AMP, early-stop on **val macro-AUROC** with patience 6
- Heavy augmentation (vertical flip + 30° rotation + color jitter) — fundus has no canonical orientation

**Runtime estimate:** ~1.5–2 hr on Kaggle T4 (smaller than Kermany but BCE is more expensive per step).

## Cell 1 — Imports + path verification

In [ ]:
import os, sys, json, time, random, pathlib
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

BRSET_ROOT = "/kaggle/input/datasets/samarthmishra208/brset-brazilian-multilabel-ophthalmological/a-brazilian-multilabel-ophthalmological-dataset-brset-1.0.1"
LABELS_CSV = os.path.join(BRSET_ROOT, "labels_brset.csv")
IMAGES_DIR = os.path.join(BRSET_ROOT, "fundus_photos")

WORK = pathlib.Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)
CKPT = RESULTS / "best.pt"

print("Torch:", torch.__version__, "GPU:", torch.cuda.is_available(),
      "Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for p in [LABELS_CSV, IMAGES_DIR]:
    assert os.path.exists(p), f"Missing: {p}"
print("Paths OK.")

## Cell 2 — Rebuild patient-disjoint index (idempotent)

Replicates the data-prep work in this single notebook so the training run is self-contained.

In [ ]:
DISEASE_COLS = [
    "diabetic_retinopathy", "macular_edema", "scar", "nevus", "amd",
    "vascular_occlusion", "hypertensive_retinopathy", "drusens",
    "hemorrhage", "myopic_fundus", "increased_cup_disc", "other",
]  # 12 labels; retinal_detachment dropped (only 7 positives)
N_LABELS = len(DISEASE_COLS)

INDEX_CSV = WORK / "brset_index.csv"
if INDEX_CSV.exists():
    df = pd.read_csv(INDEX_CSV)
    print(f"Loaded existing index: {len(df):,} rows")
else:
    df = pd.read_csv(LABELS_CSV)
    patient_icdr = df.groupby("patient_id")["DR_ICDR"].max()
    rng = np.random.default_rng(SEED)
    pat2split = {}
    for icdr in sorted(patient_icdr.unique()):
        patients = sorted(patient_icdr[patient_icdr == icdr].index.tolist())
        rng.shuffle(patients)
        n = len(patients)
        n_train, n_val = int(round(0.70 * n)), int(round(0.15 * n))
        for i, pid in enumerate(patients):
            if i < n_train:                pat2split[pid] = "train"
            elif i < n_train + n_val:      pat2split[pid] = "val"
            else:                          pat2split[pid] = "test"
    df["split"] = df["patient_id"].map(pat2split)
    df["abs_path"] = df["image_id"].apply(lambda x: os.path.join(IMAGES_DIR, f"{x}.jpg"))
    bad = df.groupby("patient_id")["split"].nunique()
    assert (bad <= 1).all()
    df.to_csv(INDEX_CSV, index=False)
    print(f"Built index: {len(df):,} rows")

print("\nSplit sizes:")
print(df["split"].value_counts())

## Cell 3 — Multilabel Dataset + transforms

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def train_tf(size=IMG_SIZE):
    return transforms.Compose([
        transforms.Resize((size + 32, size + 32)),
        transforms.RandomCrop(size),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def eval_tf(size=IMG_SIZE):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class BRSETDataset(Dataset):
    def __init__(self, index_csv, split, label_cols, transform=None):
        full = pd.read_csv(index_csv)
        self.df = full[full["split"] == split].reset_index(drop=True)
        self.label_cols = label_cols
        self.transform = transform
        self.labels = self.df[label_cols].astype(np.float32).to_numpy()  # (N, 12)

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["abs_path"]).convert("RGB")
        if self.transform: img = self.transform(img)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        # Return demographics for later audit (sex, camera, quality, age, exam_eye)
        return (img, y, str(row["patient_id"]),
                int(row.get("patient_sex", -1)),
                str(row.get("camera", "")),
                str(row.get("quality", "")),
                float(row["patient_age"]) if pd.notna(row.get("patient_age")) else -1.0)


print("Dataset + transforms defined.")

## Cell 4 — Build loaders + per-label pos_weight

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2

train_ds = BRSETDataset(str(INDEX_CSV), "train", DISEASE_COLS, transform=train_tf())
val_ds   = BRSETDataset(str(INDEX_CSV), "val",   DISEASE_COLS, transform=eval_tf())
test_ds  = BRSETDataset(str(INDEX_CSV), "test",  DISEASE_COLS, transform=eval_tf())
print(f"train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}")

# Per-label pos_weight = N_neg / N_pos  — the standard BCE rebalancing trick
n_pos = train_ds.labels.sum(axis=0)
n_neg = len(train_ds) - n_pos
pos_weight = torch.tensor(n_neg / np.maximum(n_pos, 1), dtype=torch.float32)
print("\nPer-label pos_weight (n_neg / n_pos):")
for d, w in zip(DISEASE_COLS, pos_weight.tolist()):
    print(f"  {d:30s}  pos={int(n_pos[DISEASE_COLS.index(d)]):4d}  pos_weight={w:7.2f}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Sanity batch
imgs, labels, pids, sexes, cams, quals, ages = next(iter(train_loader))
print(f"\nBatch: imgs={imgs.shape}  labels={labels.shape}  "
      f"range=[{imgs.min():.2f},{imgs.max():.2f}]")
print(f"Mean label vector (train batch): {labels.mean(dim=0).numpy().round(3).tolist()}")

## Cell 5 — Build ResNet-50 with 12-class sigmoid head

In [ ]:
def build_resnet50(num_labels=N_LABELS, dropout=0.3):
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    m = models.resnet50(weights=weights)
    in_feat = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_feat, num_labels))
    return m

model = build_resnet50().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-50 multilabel: {n_params/1e6:.2f}M params, {N_LABELS}-class sigmoid head")

# Sanity forward pass
model.eval()
with torch.no_grad():
    out = model(imgs.to(DEVICE))
print(f"Forward pass OK. Output shape: {out.shape}  (expected: [{BATCH_SIZE}, {N_LABELS}])")
print(f"Output range (raw logits, before sigmoid): [{out.min():.2f}, {out.max():.2f}]")

## Cell 6 — Training loop

Multilabel BCE-with-logits + per-label `pos_weight`. Monitor val macro-AUROC. Patience 6.

In [ ]:
EPOCHS = 25
LR = 1e-4
WD = 1e-4
WARMUP_EPOCHS = 2
PATIENCE = 6

@torch.no_grad()
def evaluate(model, loader, device, return_full=False):
    model.eval()
    losses, all_logits, all_labels = [], [], []
    all_pids, all_sexes, all_cams, all_quals, all_ages = [], [], [], [], []
    crit = nn.BCEWithLogitsLoss()
    for x, y, pid, sex, cam, qual, age in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x); loss = crit(logits, y)
        losses.append(loss.item() * x.size(0))
        all_logits.append(logits.float().cpu()); all_labels.append(y.cpu())
        all_pids.extend(list(pid))
        all_sexes.extend([int(s) for s in sex])
        all_cams.extend(list(cam)); all_quals.extend(list(qual))
        all_ages.extend([float(a) for a in age])
    n = sum(t.size(0) for t in all_labels)
    avg_loss = sum(losses) / n
    logits = torch.cat(all_logits); labels = torch.cat(all_labels)
    probs = torch.sigmoid(logits).numpy()
    y_true = labels.numpy()
    # Per-label AUROC where at least one positive and one negative exist
    per_label_auroc = {}
    aurocs = []
    for i, d in enumerate(DISEASE_COLS):
        y_bin = y_true[:, i]
        if 0 < y_bin.sum() < len(y_bin):
            a = roc_auc_score(y_bin, probs[:, i])
            per_label_auroc[d] = float(a); aurocs.append(a)
        else:
            per_label_auroc[d] = None
    macro_auroc = float(np.mean(aurocs)) if aurocs else float('nan')
    if return_full:
        return (avg_loss, macro_auroc, per_label_auroc, probs, y_true,
                np.array(all_pids), np.array(all_sexes), np.array(all_cams),
                np.array(all_quals), np.array(all_ages))
    return avg_loss, macro_auroc, per_label_auroc

def cosine_warmup_lr(epoch, total, warmup, base):
    if epoch < warmup: return base * (epoch + 1) / warmup
    p = (epoch - warmup) / max(1, total - warmup)
    return base * 0.5 * (1 + np.cos(np.pi * p))

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

best_auroc = -1.0; best_epoch = -1; no_improve = 0
history = []

for epoch in range(EPOCHS):
    lr_now = cosine_warmup_lr(epoch, EPOCHS, WARMUP_EPOCHS, LR)
    for g in optimizer.param_groups: g["lr"] = lr_now
    model.train()
    t0 = time.time(); running = 0.0; n_seen = 0
    for x, y, *_ in train_loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x); loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        running += loss.item() * x.size(0); n_seen += x.size(0)
    train_loss = running / n_seen
    val_loss, val_auroc, _ = evaluate(model, val_loader, DEVICE)
    dur = time.time() - t0
    history.append({"epoch": epoch, "lr": lr_now, "train_loss": train_loss,
                    "val_loss": val_loss, "val_macro_auroc": val_auroc, "seconds": dur})
    print(f"Epoch {epoch+1:02d}/{EPOCHS}  lr={lr_now:.2e}  train_loss={train_loss:.4f}  "
          f"val_loss={val_loss:.4f}  val_macro_auroc={val_auroc:.4f}  ({dur:.0f}s)")
    if val_auroc > best_auroc:
        best_auroc = val_auroc; best_epoch = epoch; no_improve = 0
        torch.save({"model_state": model.state_dict(), "epoch": epoch,
                    "val_macro_auroc": val_auroc, "labels": DISEASE_COLS}, CKPT)
        print(f"  -> new best val_macro_auroc={val_auroc:.4f}, saved {CKPT}")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stop at epoch {epoch+1}"); break

pd.DataFrame(history).to_csv(RESULTS / "training_history.csv", index=False)
print(f"\nBest val_macro_auroc={best_auroc:.4f} at epoch {best_epoch+1}")

## Cell 7 — Test eval with bootstrap CIs + per-label metrics

In [ ]:
state = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model_state"]); model.eval()
print(f"Loaded checkpoint epoch {state['epoch']+1}  val_macro_auroc={state['val_macro_auroc']:.4f}")

(test_loss, test_macro_auroc, per_label_auroc, test_probs, test_y,
 test_pids, test_sex, test_cam, test_qual, test_age) = evaluate(model, test_loader, DEVICE, return_full=True)

# Per-label bootstrap CI
def bootstrap_per_label(probs, y_true, n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    out = {}
    for i, d in enumerate(DISEASE_COLS):
        aurocs, auprcs = [], []
        for _ in range(n_boot):
            idx = rng.integers(0, n, size=n)
            yb = y_true[idx, i]; pb = probs[idx, i]
            if 0 < yb.sum() < len(yb):
                aurocs.append(roc_auc_score(yb, pb))
                auprcs.append(average_precision_score(yb, pb))
        if aurocs:
            out[d] = {
                "n_pos": int(y_true[:, i].sum()),
                "auroc": float(np.mean(aurocs)),
                "auroc_ci95": [float(np.quantile(aurocs, alpha/2)),
                               float(np.quantile(aurocs, 1-alpha/2))],
                "auprc": float(np.mean(auprcs)),
                "auprc_ci95": [float(np.quantile(auprcs, alpha/2)),
                               float(np.quantile(auprcs, 1-alpha/2))],
            }
        else:
            out[d] = None
    return out

print("\n=== Per-label test AUROC + bootstrap 95% CIs ===")
per_label_full = bootstrap_per_label(test_probs, test_y)
for d, m in per_label_full.items():
    if m is None:
        print(f"  {d:30s}: insufficient positives")
    else:
        print(f"  {d:30s}: n_pos={m['n_pos']:4d}  AUROC={m['auroc']:.4f}  "
              f"CI={m['auroc_ci95']}  AUPRC={m['auprc']:.4f}")

print(f"\nMacro AUROC across all 12 labels: {test_macro_auroc:.4f}")

results = {
    "test_n": int(len(test_y)),
    "test_macro_auroc": float(test_macro_auroc),
    "per_label": per_label_full,
    "labels": DISEASE_COLS,
    "best_epoch": int(state["epoch"] + 1),
    "best_val_macro_auroc": float(state["val_macro_auroc"]),
}
with open(RESULTS / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

## Cell 8 — Fairness audit by sex, camera, quality, age band

Run the per-label AUROC analysis again, but stratified by demographic subgroup. Sample sizes shrink so CIs will be wider; we report n_pos per subgroup so we know which numbers to trust.

In [ ]:
def per_label_auroc_subgroup(probs, y_true, mask, name, group_val):
    if mask.sum() < 100:  # too few samples for meaningful audit
        return None
    p, y = probs[mask], y_true[mask]
    out = {"subgroup": f"{name}={group_val}", "n": int(mask.sum()), "per_label": {}}
    for i, d in enumerate(DISEASE_COLS):
        yb = y[:, i]; pb = p[:, i]
        if 0 < yb.sum() < len(yb) and yb.sum() >= 10:
            out["per_label"][d] = {
                "n_pos": int(yb.sum()),
                "auroc": float(roc_auc_score(yb, pb)),
                "auprc": float(average_precision_score(yb, pb)),
            }
        else:
            out["per_label"][d] = None
    return out

audit = {}

# By sex
audit["by_sex"] = []
for s in [1, 2]:
    res = per_label_auroc_subgroup(test_probs, test_y, test_sex == s,
                                    "sex", "male" if s == 1 else "female")
    if res: audit["by_sex"].append(res)

# By camera
audit["by_camera"] = []
for c in sorted(set(test_cam)):
    if not c: continue
    res = per_label_auroc_subgroup(test_probs, test_y, test_cam == c, "camera", c)
    if res: audit["by_camera"].append(res)

# By quality
audit["by_quality"] = []
for q in sorted(set(test_qual)):
    if not q: continue
    res = per_label_auroc_subgroup(test_probs, test_y, test_qual == q, "quality", q)
    if res: audit["by_quality"].append(res)

# By age band (only where age is known)
audit["by_age_band"] = []
known_age = test_age >= 0
age_bands = [(0, 40, "<40"), (40, 60, "40-60"), (60, 80, "60-80"), (80, 200, "80+")]
for lo, hi, name in age_bands:
    mask = known_age & (test_age >= lo) & (test_age < hi)
    res = per_label_auroc_subgroup(test_probs, test_y, mask, "age", name)
    if res: audit["by_age_band"].append(res)

# Print a compact summary: per-disease AUROC across subgroups for the high-prevalence labels
def summary_table(audit_key, group_name):
    print(f"\n=== Audit by {group_name} ===")
    rows = audit[audit_key]
    if not rows:
        print("  (no usable subgroups)"); return
    high_prev = ["diabetic_retinopathy", "drusens", "increased_cup_disc",
                 "macular_edema", "amd", "scar"]
    header = f"  {'disease':28s}  " + "  ".join(f"{r['subgroup']:>20s}" for r in rows)
    print(header)
    for d in high_prev:
        cells = []
        for r in rows:
            v = r["per_label"].get(d)
            cells.append(f"AUROC={v['auroc']:.3f} (n_pos={v['n_pos']})" if v else "—")
        print(f"  {d:28s}  " + "  ".join(f"{c:>20s}" for c in cells))

summary_table("by_sex", "sex")
summary_table("by_camera", "camera")
summary_table("by_quality", "quality")
summary_table("by_age_band", "age band")

# Save the full audit JSON
with open(RESULTS / "fairness_audit.json", "w") as f:
    json.dump(audit, f, indent=2)
print(f"\nFull audit saved to {RESULTS/'fairness_audit.json'}")

## Cell 9 — Persist all artifacts

In [ ]:
import shutil
zip_path = WORK / "brset_baseline_results.zip"
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(zip_path.with_suffix(""), "zip", root_dir=str(RESULTS))
print(f"Wrote {zip_path}  ({zip_path.stat().st_size/1e6:.1f} MB)")
print("Download from Notebook page → Output tab → brset_baseline_results.zip")